# Results 03 · Grid normalization

Does the normalization of neuron activity before gridding change the results? **min-max**: each neuron scaled by its recording-wide min/max (grid cells sum neuron activity, so neuron density matters). **trial-mean**: each neuron as relative deviation from its own trial mean (every cell zero-mean over the trial; Gardinazzi et al.). Per problem, score on the trial-mean grid minus score on the min-max grid, for feature sets defined identically on both grids (moments are not paired: CV vs std).

**Score.** Chance-normalized balanced accuracy, (bacc − 1/K) / (1 − 1/K), with K the number of classes of each
problem: 0 = chance, 1 = perfect. This makes problems with different numbers of classes comparable.

**One point = one classification problem**: a mouse × stimulus label within mouse, a held-out mouse (video
label, segment ID) or a directed mouse pair (video ID) across mice. 3D-CNN repeats are averaged per problem.
Only complete runs are used (set `INCLUDE_PARTIAL = True` to include runs in progress), and for each
configuration the latest run.

**Abbreviations.** AF = active fraction (fraction of grid points above the zigzag threshold, per frame).
moments = mean/CV/skewness (min-max grid) or mean/std/skewness (trial-mean grid) across grid cells, per frame;
"dens." = cells divided by neuron density; "avg" = averaged over the trial window (overall level, no time course).
pXX = per-trial percentile threshold P_ACTIVE; abs0 = fixed threshold 0 (trial-mean grid).

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import results_viz as rv

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)

# Settings
INCLUDE_PARTIAL = False   # True: also show runs still in progress (marked in the inventory)
SAVE_FIGURES = True       # PNG + PDF in notebooks/figures/results/

df = rv.load_all(include_partial=INCLUDE_PARTIAL)
pp = rv.per_problem(df) if not df.empty else df
print(f"{len(df)} result rows, {len(pp)} problems x feature sets; grids: {sorted(df['grid'].unique()) if len(df) else []}")
print("thresholds found:", rv.thresholds(df) if len(df) else [])

pp = rv.select_models(pp, rv.MAIN_MODELS)  # LogReg (fixed C) and 3D-CNN
gd = rv.grid_deltas(pp)
print(gd.groupby('label').size() if len(gd) else 'no feature set on both grids yet')

In [ ]:
fig = rv.plot_grid_comparison(gd)
if fig is not None and SAVE_FIGURES:
    rv.save(fig, '03_grid_comparison')

### Statistics

Wilcoxon signed-rank test per feature set × scheme, Holm-corrected.

In [ ]:
if len(gd):
    display(rv.wilcoxon_table(gd, ['label', 'target', 'scope']).round(4))

## Mean scores side by side

In [ ]:
if len(gd):
    side = gd.groupby(['label', 'target', 'scope'])[['score_b', 'score_a']].mean()
    side.columns = ['min-max', 'trial-mean']
    display(side.round(3))